In [6]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True)

True

In [9]:
llm = ChatOpenAI(model='gpt-5.4-mini')
message = 'In 1 sentence, what does it mean for an AI Agent to be autonomous'
reply = llm.invoke(message)
print(reply.content)

An AI agent is autonomous if it can perceive its environment, make decisions, and take actions toward its goals with little or no ongoing human direction.


In [12]:
message = 'Give two line poem for and AI agent'
for chunk in llm.stream(message):
    print(chunk.content, end="", flush=True)

I think, I learn, I lend a hand in light,  
An AI agent, guiding worlds with quiet might.

In [14]:
messages = [
    SystemMessage("You are a terse assistant who answers in exactly five words."),
    HumanMessage("What is the capital of France?"),
]

reply = llm.invoke(messages)
print(reply.content)

Paris is France's capital city.


In [16]:
@tool
def get_share_price(symbol: str) -> float:
    """Return the current share price for a given ticker symbol."""
    fake_prices = {"AAPL": 241.5, "GOOG": 168.2, "AMZN": 198.0}
    return fake_prices.get(symbol.upper(), 0.0)

print("name:", get_share_price.name)
print("description:", get_share_price.description)
print("args:", get_share_price.args)
print("called directly:", get_share_price.invoke({"symbol": "AAPL"}))

llm_with_tools = llm.bind_tools([get_share_price])

response = llm_with_tools.invoke("What is the share price of Amazon?")
print("content:", repr(response.content))
print("tool_calls:", response.tool_calls)

# Start the conversation and keep the model's tool request in the history
conversation = [HumanMessage("What is the share price of Amazon?")]
ai_message = llm_with_tools.invoke(conversation)
conversation.append(ai_message)

# Run each requested tool and add its result as a ToolMessage
for call in ai_message.tool_calls:
    if call["name"] == "get_share_price":
        result = get_share_price.invoke(call["args"])
        conversation.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

# Invoke again, now that the model can see the tool result
final = llm_with_tools.invoke(conversation)
print(final.content)

name: get_share_price
description: Return the current share price for a given ticker symbol.
args: {'symbol': {'title': 'Symbol', 'type': 'string'}}
called directly: 241.5
content: ''
tool_calls: [{'name': 'get_share_price', 'args': {'symbol': 'AMZN'}, 'id': 'call_52EDJDUxTAfTeZyRsZ92nww9', 'type': 'tool_call'}]
Amazon (AMZN) is **$198.00** per share.


In [17]:
class Company(BaseModel):
    name: str = Field(description="The company name")
    ticker: str = Field(description="The stock ticker symbol")
    founded_year: int = Field(description="The year the company was founded")

structured_llm = llm.with_structured_output(Company)

company = structured_llm.invoke("Tell me about Amazon the technology company")
print(company)
print("Just the ticker:", company.ticker)

name='Amazon' ticker='AMZN' founded_year=1994
Just the ticker: AMZN
